# LightGBM Volatility Backtest

CPU-only tree-based walk-forward backtest. LightGBM handles NaN natively,
so no scaling or missing-value imputation is needed for features.

**Architecture (post Step 9 dedup).** This notebook now contains only
LightGBM-specific glue:
* `DEFAULT_LGBM_PARAMS` — method defaults merged with `--params-file`.
* `fit_predict_lgbm` — closure over `run_backtest(use_scaling=False)`.
* `main()` — three-line wrapper around `parse_executor_args` +
  `run_executor`.

All dataset loading, transforms, HAR/calendar features, horizon shift,
chunk slicing, smearing, and reduce-JSON writing live in
`src/executor.py` (exported from `notebooks/pipeline/05_executor.ipynb`).

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow tqdm lightgbm

import sys  # noqa: E402

sys.path.insert(0, "/content/harxhar")

In [ ]:
# ── Notebook-only demo parameters ──
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
REFIT_FREQUENCY = 5
DATA_PATH = "all30min"

## Module imports + defaults

First export cell: docstring, imports, default-hyperparam dict.

In [ ]:
# export
"""LightGBM volatility backtest executor.

Method-specific glue around the shared scaffold in :mod:`src.executor`.
Only the model factory + default hyperparams + ``main()`` wrapper live
here; everything else (CLI parsing, loading, features, backtest loop,
smearing, reduce JSON) is owned by ``src.executor``.
"""

from __future__ import annotations

import json

import numpy as np
from lightgbm import LGBMRegressor

from src.executor import parse_executor_args, run_executor
from src.loading import parse_exog_cols
from src.scaling import run_backtest

# Per-method default hyperparams. Tuned overrides from --params-file are
# merged on top via dict.update().
DEFAULT_LGBM_PARAMS: dict = dict(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.1,
    n_jobs=-1,
    verbose=-1,
)

# Method-specific refit-frequency default. The CLI ``--refit-frequency``
# sentinel of None falls back to this; the original ml_lightgbm.py used
# 1 as its argparse default.
DEFAULT_LGBM_REFIT_FREQUENCY: int = 1

In [ ]:
# ── Imports smoke test ──
_expected = [
    "json",
    "np",
    "LGBMRegressor",
    "parse_executor_args",
    "run_executor",
    "parse_exog_cols",
    "run_backtest",
    "DEFAULT_LGBM_PARAMS",
    "DEFAULT_LGBM_REFIT_FREQUENCY",
]
_missing = [n for n in _expected if n not in dir()]
assert not _missing, f"missing imports: {_missing}"
print(f"imports OK ({len(_expected)} symbols)")
print(f"DEFAULT_LGBM_PARAMS={DEFAULT_LGBM_PARAMS}")

## `fit_predict_lgbm` — the only model-specific call

Closure over `run_backtest` with `use_scaling=False` (LightGBM handles
feature scale natively). Each call to the closure builds a fresh
model-factory bound to the merged hyperparams, then runs the
shared walk-forward loop.

**Why `model_fn` not `fit_predict` directly?** `run_backtest` already
implements the rolling-window refit logic; we only need to vary the
factory.

`random_state` is sourced from the merged hyperparams dict (default
42) so every `LGBMRegressor` instance is deterministic.

In [ ]:
# export
def fit_predict_lgbm(
    X_chunk: np.ndarray,
    y_chunk: np.ndarray,
    train_win_periods: int,
    hyperparams: dict,
) -> np.ndarray:
    """Walk-forward backtest with LightGBM. Returns OOS predictions.

    Wraps :func:`src.scaling.run_backtest` with the LGBM-specific
    settings (``use_scaling=False``; refit frequency from
    ``hyperparams['_refit_frequency']``). The model is constructed
    with ``random_state=hyperparams.get("random_state", 42)`` for
    reproducibility.
    """
    refit_frequency = int(hyperparams.get("_refit_frequency", DEFAULT_LGBM_REFIT_FREQUENCY))
    # Strip our internal control key before passing to LGBMRegressor.
    model_kwargs = {k: v for k, v in hyperparams.items() if not k.startswith("_")}
    model_kwargs.setdefault("random_state", 42)

    def model_fn():
        return LGBMRegressor(**model_kwargs)

    return run_backtest(
        model_fn,
        X_chunk,
        y_chunk,
        train_win=train_win_periods,
        refit_frequency=refit_frequency,
        use_scaling=False,
    )

In [ ]:
# ── Smoke-test fit_predict_lgbm on synthetic data ──
rng = np.random.default_rng(42)
n, k = 600, 4
X_demo = rng.normal(size=(n, k))
y_demo = rng.normal(size=n)
_hp = dict(DEFAULT_LGBM_PARAMS, _refit_frequency=10, n_estimators=20, random_state=42)  # tiny for speed
preds = fit_predict_lgbm(X_demo, y_demo, train_win_periods=500, hyperparams=_hp)
assert preds.shape == (n - 500,), preds.shape
assert np.all(np.isfinite(preds)), "predictions must be finite"
print(f"fit_predict_lgbm OK: preds.shape={preds.shape}, mean={preds.mean():.4f}")

## In-notebook demo (manual pipeline)

A didactic walk-through of the full pipeline using the in-notebook
constants above. The exported `main()` re-implements this via
`run_executor`; this section is here so a reader can see each step.

In [ ]:
# ── Load + transform ──
import numpy as np

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH, allow_missing=True)
print(f"Loaded: {df.shape}")

# Full transform on RV target (diurnal + winsor)
adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"adj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# --- Features from shared pipeline ---
from src.transforms import resolve_har_lags, generate_har_features, add_calendar_features

df, har_names = generate_har_features(df, target_col="adj_RV")
cal_names = add_calendar_features(df)
feature_names = har_names + cal_names

print(f"HAR lags: {resolve_har_lags()}")
print(f"Features ({len(feature_names)}): {feature_names}")

# Drop initial NaN rows from HAR computation
max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)
print(f"Shape after lag trim: {df.shape}")

In [ ]:
# --- LightGBM walk-forward backtest (demo) ---
from lightgbm import LGBMRegressor
from src.transforms import apply_horizon_shift
from src.scaling import run_backtest
from src.transforms import PERIODS_PER_DAY

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

# Horizon shift
X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Samples: {len(y)}, Features: {X.shape[1]}, Train window: {train_win}")

model_fn = lambda: LGBMRegressor(**DEFAULT_LGBM_PARAMS, random_state=42)

predictions = run_backtest(model_fn, X, y, train_win=train_win, refit_frequency=REFIT_FREQUENCY, use_scaling=False)
print(f"Predictions: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}")

# Fit final model on last window for feature importance plot
model = model_fn()
model.fit(X[-train_win:], y[-train_win:])

In [ ]:
# ── Feature importance ──
import matplotlib.pyplot as plt

importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx])
ax.set_xlabel("Feature Importance (split count)")
ax.set_title("LightGBM Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation ---
from src.evaluation import apply_duan_smearing, calculate_metrics
import pandas as pd

oos_start = train_win
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(predictions, y_oos, baselines_oos)

results_df = pd.DataFrame(
    {
        "date": dates_oos,
        "horizon": HORIZON,
        "true_adj": y_oos,
        "pred_adj": predictions,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    }
)

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

## `main()` — CLI entry point

Three responsibilities:
1. Parse the canonical CLI via `parse_executor_args`.
2. Merge `--params-file` JSON onto `DEFAULT_LGBM_PARAMS`, and inject
   `random_state=args.seed` (default 42).
3. Hand off to `run_executor` with method-specific toggles
   (`add_calendar=True`, `target_use_diurnal=True`,
   `target_winsor_window=240`, `dropna_with_exog=False`).

The `--segment` / `--lag-scope` flags are present in the parser for
tuner-CLI uniformity but are passed through unused (LightGBM runs only
in segment-less / `lag_scope='global'` mode).

In [ ]:
# export
def main() -> None:
    args = parse_executor_args("LightGBM walk-forward backtest")

    tuned_params: dict = {}
    if args.params_file:
        with open(args.params_file) as f:
            tuned_params = json.load(f)

    # Method defaults <- tuned overrides; refit-frequency from CLI sentinel.
    hyperparams = dict(DEFAULT_LGBM_PARAMS)
    hyperparams.update(tuned_params)
    hyperparams["random_state"] = args.seed
    refit_frequency = args.refit_frequency if args.refit_frequency is not None else DEFAULT_LGBM_REFIT_FREQUENCY
    hyperparams["_refit_frequency"] = refit_frequency

    exog_cols = parse_exog_cols(args.exog_cols)

    run_executor(
        method_name="lightgbm",
        fit_predict=fit_predict_lgbm,
        hyperparams=hyperparams,
        data_path=args.data_path,
        output_file=args.output_file,
        horizon=args.horizon,
        train_window=args.train_window,
        start=args.start,
        end=args.end,
        exog_cols=exog_cols,
        segment=args.segment,
        lag_scope=args.lag_scope,
        add_calendar=True,
        target_use_diurnal=True,
        target_winsor_window=240,
        dropna_with_exog=False,
        seed=args.seed,
    )

In [ ]:
# ── Smoke-test main() signature ──
import inspect

assert callable(main)
sig = inspect.signature(main)
assert list(sig.parameters) == [], f"main() should take no args; got {sig}"
print("main signature OK:", sig)

In [ ]:
# export
if __name__ == "__main__":
    main()

In [ ]:
from _exporter import export_notebook

export_notebook("ml_lightgbm.ipynb", "../src/ml_lightgbm.py")